# 4. Validate the FTH forward calculation

Check internal consistency, repeatable acquisition and numerical sensitivity. The independent analytic/padded-grid RS checks live in the production tests listed in [VALIDATION.md](../VALIDATION.md). Numerical consistency is distinct from experimental accuracy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


In [ ]:
# EDIT HERE: sample dimensions are nm; detector lengths are m.
sample = ex.FTHConfig(
    energy_eV=778., n=192, dx_nm=10.,
    layers_nm=(('Au',300.), ('SiN',20.), ('Pt',2.), ('Co',10.), ('Pt',2.)),
    max_slice_nm=5., object_radius_nm=180., reference_radius_nm=30.,
    reference_xy_nm=(600.,0.), beam_sigma_nm=650.,
    domain_period_nm=110., propagate=True,
    energies_eV=(772.,775.,778.,780.,783.,790.,795.,800.),
)
camera = wf.DetectorGeometry(shape=(128,128), pixel_size_m=13.5e-6,
                             distance_m=.05, method='fraunhofer')
OUT = PAPER / 'results' / 'notebook_fth_validation'


In [ ]:
metrics = wf.validate(OUT, sample)
for key, value in metrics.items():
    print(f'{key}: {value}')


## Transverse refinement at fixed field of view
Halve pitch while doubling pixel count. A common-q FFT cube then has the same reciprocal increment; compare the central overlapping q support. Aperture discretization and domain sampling can affect this result. This optional refinement is more expensive than the baseline.

In [ ]:
coarse = ex.fth_case(config=sample)
fine = ex.fth_case(config=replace(sample, n=2*sample.n, dx_nm=sample.dx_nm/2))
start = (fine['difference'].shape[0]-sample.n)//2
matched = fine['difference'][start:start+sample.n, start:start+sample.n]/4
# The orthonormal 2D FFT intensity grows by four when dx halves at fixed area.
error = np.linalg.norm(matched-coarse['difference'])/np.linalg.norm(coarse['difference'])
print('Transverse refinement relative L2 (common q support):', error)


## Remaining validation
Enlarge the source window at fixed pitch, refine material slices again, refine detector footprints and the reference aperture, and converge the observable you intend to use. Check optical constants and compare with independent theory or measurement where available. A single refinement difference is not an error bound.